# Module 04.03 — Legacy Visualizations (`visualization`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.3 Legacy Visualizations (`visualization`)

The `visualization` type pre-dates Lens. It covers **Aggregation-based charts** (bar,
line, pie, data table, metric, gauge, goal), **TSVB** (Time Series Visual Builder),
**Timelion**, and **Vega/Vega-Lite** visualizations. All of these still use the `visualization`
saved object type — even in modern Kibana versions.

The distinguishing feature of the schema is the `visState` attribute: a JSON string
(JSON embedded inside JSON) that carries the entire visualization definition — the
`type` field tells Kibana which renderer to use, and `params` contains the
renderer-specific configuration. Because `visState` is opaque from Kibana's
perspective, its structure varies wildly between chart types. TSVB stores time-series
panel configs; Vega stores raw Vega-Lite specs; aggregation charts store a `aggs` array.

For snapshot purposes, legacy visualizations behave like any other saved object — they
are fully captured in the Kibana feature state. The one subtlety is that TSVB and
aggregation-based charts reference a Data View via the `references` array, while Vega
charts that embed their own ES queries may not — making Vega workpads closer to
Canvas workpads in their self-contained portability.

### Create in Kibana UI
1. Go to **Visualize Library → Create visualization**
2. Choose **Aggregation based → Vertical bar**
3. Select the eCommerce data view
4. Y-axis: `Sum` of `taxful_total_price`
5. X-axis: `Date histogram` on `order_date`
6. Save as `Training — Revenue by Day`

In [ ]:
heading('4.3 Legacy Visualization — inspect sample data objects')

vizzes = find_saved_objects('visualization')
console.print(f'  Found {len(vizzes)} visualization(s)')
if vizzes:
    v = vizzes[0]
    pp(v, 'visualization saved object')
    info('Key fields in attributes:')
    info('  visState        — JSON blob: vis type, params, aggs')
    info('  uiStateJSON     — panel-level UI state')
    info('  kibanaSavedObjectMeta.searchSourceJSON — linked data view')
    info('references       — links to index-pattern and any saved search')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/visualize', 'Visualize Library — browse legacy visualizations')

In [ ]:
heading('4.3 Legacy Visualization — create via API')

ecom_dv = next((o for o in find_saved_objects('index-pattern') if 'ecommerce' in o['attributes'].get('title', '')), None)
if not ecom_dv:
    raise RuntimeError('eCommerce data view not found — re-run 00_setup')
ecom_dv_id = ecom_dv['id']

existing_viz = next((o for o in find_saved_objects('visualization') if o['attributes'].get('title') == 'Training — Order Count'), None)
if existing_viz:
    viz_id = existing_viz['id']
    info(f'Visualization already exists: id={viz_id}')
else:
    viz_resp = kibana_post('/api/saved_objects/visualization', {
        'attributes': {
            'title': 'Training — Order Count',
            'visState': json.dumps({
                'title': 'Training — Order Count',
                'type': 'metric',
                'params': {
                    'addTooltip': True, 'addLegend': False, 'type': 'metric',
                    'metric': {'percentageMode': False, 'useRanges': False,
                               'colorSchema': 'Green to Red', 'metricColorMode': 'None',
                               'colorsRange': [{'from': 0, 'to': 10000}],
                               'labels': {'show': True}, 'invertColors': False,
                               'style': {'bgFill': '#000', 'bgColor': False,
                                         'labelColor': False, 'subText': '', 'fontSize': 60}}
                },
                'aggs': [{'id': '1', 'enabled': True, 'type': 'count', 'schema': 'metric', 'params': {}}]
            }),
            'uiStateJSON': '{}',
            'description': '',
            'kibanaSavedObjectMeta': {
                'searchSourceJSON': json.dumps({
                    'query': {'query': '', 'language': 'kuery'},
                    'filter': [],
                    'indexRefName': 'kibanaSavedObjectMeta.searchSourceJSON.index',
                })
            }
        },
        'references': [{'name': 'kibanaSavedObjectMeta.searchSourceJSON.index', 'type': 'index-pattern', 'id': ecom_dv_id}],
    })
    viz_id = viz_resp['id']
    success(f'Legacy visualization created: id={viz_id}')

obj = kibana_get(f'/api/saved_objects/visualization/{viz_id}')
pp(obj, 'visualization saved object')
info('Key fields:')
info('  visState  — JSON blob: chart type, aggregation config, display params')
info('  references — linked data view (index-pattern)')
kibana_link(f'/app/visualize#/edit/{viz_id}', 'Open Training — Order Count in Visualize')

In [ ]:
# Ensure visualization exists before snapshotting (re-runnable)
if not any(o['id'] == viz_id for o in find_saved_objects('visualization')):
    warn('Visualization missing — recreating')
    ecom_dv_id = next(o['id'] for o in find_saved_objects('index-pattern') if 'ecommerce' in o['attributes'].get('title', ''))
    viz_resp = kibana_post('/api/saved_objects/visualization', {
        'attributes': {
            'title': 'Training — Order Count',
            'visState': json.dumps({'title': 'Training — Order Count', 'type': 'metric',
                'params': {'addTooltip': True, 'addLegend': False, 'type': 'metric',
                    'metric': {'percentageMode': False, 'useRanges': False, 'colorSchema': 'Green to Red',
                               'metricColorMode': 'None', 'colorsRange': [{'from': 0, 'to': 10000}],
                               'labels': {'show': True}, 'invertColors': False,
                               'style': {'bgFill': '#000', 'bgColor': False, 'labelColor': False, 'subText': '', 'fontSize': 60}}},
                'aggs': [{'id': '1', 'enabled': True, 'type': 'count', 'schema': 'metric', 'params': {}}]}),
            'uiStateJSON': '{}', 'description': '',
            'kibanaSavedObjectMeta': {'searchSourceJSON': json.dumps(
                {'query': {'query': '', 'language': 'kuery'}, 'filter': [],
                 'indexRefName': 'kibanaSavedObjectMeta.searchSourceJSON.index'})}
        },
        'references': [{'name': 'kibanaSavedObjectMeta.searchSourceJSON.index', 'type': 'index-pattern', 'id': ecom_dv_id}],
    })
    viz_id = viz_resp['id']

snap_delete_restore_cycle('snap-viz-test', 'Legacy Visualization')

kibana_delete(f'/api/saved_objects/visualization/{viz_id}')
warn(f'Accidentally deleted visualization {viz_id}')
assert not any(o['id'] == viz_id for o in find_saved_objects('visualization')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-viz-test')
time.sleep(3)

restored = find_saved_objects('visualization')
assert any(o['id'] == viz_id for o in restored), 'Visualization should be restored'
success(f'Visualization {viz_id} successfully restored!')
kibana_link(f'/app/visualize#/edit/{viz_id}', 'Verify restored visualization')